In [ ]:
import requests
import pandas as pd
import time
import aiohttp
import asyncio
import os
import ssl
import nest_asyncio
from tqdm.asyncio import tqdm_asyncio

: 

## APIs Used:

- Bills - https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/Documents
- Roll Call Votes - https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/Branches/{branch}/RollCalls/{rollCallNumber}
- Legislative leaders - https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/Branches/{branch}/Leadership
- Member details - https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/LegislativeMembers 
- Bill roll-calls - https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/Documents/{billNumber}/RollCalls
- Committee - https://malegislature.gov/api/GeneralCourts/{court}/Committees


In [ ]:
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

## Extracting Bills

In [ ]:
''' 
    Code to extract bill details using Senate MA Legislature API 
    https://malegislature.gov/api/GeneralCourts/{court_no}/Documents returns bill details for a given general court number 
    General Court Number representing 182 for 2001-2002, 183 for 2003-2004, ....., 194 for 2025-2026
    Includes bills for House and Senate 
'''
nest_asyncio.apply()

# Fetching JSON data from the API
async def fetch_json(session, url):
    try:
        async with session.get(url, ssl=ssl_ctx, timeout=20) as resp:
            # Handle non-success responses
            if resp.status == 404:
                print(f"No data found at {url}")
                return None

            if resp.status == 400:
                print(f"Bad request at {url}")
                return None

            if resp.status != 200:
                print(f"Error {resp.status} at {url}")
                return None

            print(f"Successfully fetched data from {url}")
            return await resp.json()

    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return None


async def scrape_bill_details():
    res = []
    courts = range(182, 195)

    async with aiohttp.ClientSession() as session:
        tasks = []

        # Create tasks for all General Court numbers (parallel)
        for court_no in courts:
            url = f"https://malegislature.gov/api/GeneralCourts/{court_no}/Documents"
            tasks.append(fetch_json(session, url))

        responses = await asyncio.gather(*tasks)

        # Process each JSON response
        for court_no, data in zip(courts, responses):
            if not data:
                continue

            # Extracting relevant fields if data exists
            for i in data:
                sponsor = i.get("PrimarySponsor") or {}

                res.append({
                    "GeneralCourtNumber": court_no,
                    "BillNumber": i.get("BillNumber"),
                    "DocketNumber": i.get("DocketNumber"),
                    "PrimarySponsorId": sponsor.get("Id"),
                    "PrimarySponsorName": sponsor.get("Name"),
                    "Title": i.get("Title")
                })

    return pd.DataFrame(res)

Bill_details = await scrape_bill_details()


In [ ]:
display(Bill_details.head())

In [ ]:
# Create parsedcsv directory if does not exist already
directory_path = "../../data/parsedcsv"

# Create the directory and any necessary parent directories
os.makedirs(directory_path, exist_ok=True)

In [ ]:
output_file='../../data/parsedcsv/MA_Bill_Details_raw.csv'
Bill_details.to_csv(output_file, index=False, encoding='utf-8')
print(f"Bill details data saved to {output_file}")

## Roll Calls- Senate

In [ ]:
'''
    Extract Senate roll calls for given session
    This code block takes Court Number, Branch, Roll call number and extracts votes for each Legislature members 
    Tracking continuous failures of 10 to stop unnecessary requests once roll calls are exhausted for a given court number
''' 

roll_calls=[]
def get_rollcalls(generalCourtNumber, branch, rollCallNumber):
    url = f'https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/Branches/{branch}/RollCalls/{rollCallNumber}'
    
    # Handling non-success responses
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()  # will raise for 400, 404, etc
        data = resp.json()
    except requests.exceptions.HTTPError as e:
        print(f'Skipping Court {generalCourtNumber}, Roll Call {rollCallNumber}: {e}')
        return None
    print(f'Successfully fetched Court {generalCourtNumber}, Roll Call {rollCallNumber}')

    # Create a dataframe storing required roll call data
    res=[]
    # Within each roll call, iterate over vote types and extract member votes 
    for i in data.keys(): 
        votes = data.get(i, [])  
        if not votes:    # Handling case where there are no votes recorded        
            continue
        if i not in [ 'Yeas','Nays','Absent']: 
            continue 
        for j in data[i]:
            res.append({
                        
                        'GeneralCourtNumber': generalCourtNumber,
                        'Branch': branch, 
                        'RollCallNumber': rollCallNumber,
                        'MemberCode':j['MemberCode'], 
                        'MemberURL':j['Details'], 
                        'Vote':i})
    return res

generalCourtNumber = range(182, 195)
branch = 'Senate'

# For rollcalls in each General court, iterate through roll calls 1 to 500 to check for existance of data. Making API call, if the roll call data exists
for court_no in generalCourtNumber:
    rollCallNumber = range(1,500)
    consecutive_failures=0 # to track continuous failures of requests. Assuming the roll calls have ended/ does not exists, moving to the next general court.

    for roll_call in rollCallNumber:
        data = get_rollcalls(court_no, branch, roll_call)

        if data:
            roll_calls.extend(data)
            consecutive_failures = 0  # reset on success
            
        else:
            consecutive_failures += 1
        # Stop if 10 consecutive failures
        if consecutive_failures >= 10:
            print(f"Stopping General Court {court_no} after 20 consecutive failed requests.")
            break

        time.sleep(0.5)
senate_votes = pd.DataFrame(roll_calls)


In [ ]:
senate_votes= pd.DataFrame(roll_calls)
senate_votes.to_csv('../../data/parsedcsv/MA_senate_rollcall_votes_raw.csv', index=False)
print("Senate roll call votes data saved to MA_senate_rollcall_votes_raw.csv")

## Legislative leaders

In [ ]:
# Extract leadership position for Senate and House members for each General Court
# "https://malegislature.gov/api/GeneralCourts/{court_no}/Branches/{branch}/Leadership" URL is used to extract leaders of a specific general court.

generalCourtNumber = range(182, 195) 
branches = ['Senate','House']
leadership = []

# Iterate through branches and general court numbers to extract leadership data
for branch in branches:
    for court_no in generalCourtNumber:
        
        url_leadership = f"https://malegislature.gov/api/GeneralCourts/{court_no}/Branches/{branch}/Leadership"
        resp_lead = requests.get(url_leadership)

        if resp_lead.status_code != 200:
            continue
        try:
            leaders = resp_lead.json()
        except ValueError:
            continue

        # Create an object leadership data    
        for leader in leaders:
            member_code = leader.get("Member", {}).get('MemberCode')
            if member_code:
                # Append leadership details to the list
                leadership.append({
                    'GeneralCourtNumber': court_no, 
                    'Branch': branch,
                    'MemberCode': member_code,
                    'Position': leader.get('Position')        
                })
        print(f"Extracted leadership for Court {court_no}, Branch {branch}")


In [ ]:
# Convert leadership data to DataFrame and save to CSV
ma_leaders= pd.DataFrame(leadership)
ma_leaders.to_csv('../../data/parsedcsv/MA_legislature_leaders.csv', index=False, encoding='utf-8')
print("Legislature leadership data saved to MA_legislature_leaders.csv")

## Members data

In [ ]:
# Member extraction for each general court
# "https://malegislature.gov/api/GeneralCourts/{court_no}/LegislativeMembers" API is used to extract member codes of all legislature members for given general court number.
# It provides member_url for each member and this code block further creates another API call to extract member details using that URL 

nest_asyncio.apply()

generalCourtNumber = range(182,195)
branches = ['Senate']
member_records = []

# Disable SSL verification to resolve SSL certificate error
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

# Function to fetch JSON data 
async def fetch_json(session, url):
    async with session.get(url, ssl=ssl_context) as resp:
        if resp.status != 200:
            return None, resp.status
        try:
            return await resp.json(), resp.status
        except:
            return None, resp.status

# Function to fetch member details
async def fetch_member_details(session, member_url, member_code, court_no):
    data, status = await fetch_json(session, member_url)

    # Handle non-success responses, storing member code with empty details 
    if status in (400, 404) or data is None:
        return {
            'GeneralCourtNumber': court_no,
            'MemberCode': member_code,
            'FullName': '',
            'District': '',
            'Branch': '',
            'Party': '',
            'Active': '',
            'LeadershipPosition': ''
        }

    # Extract relevant member details and return as a dictionary
    print(f'Successfully fetched member {member_code} for Court {court_no}')
    return {
        'GeneralCourtNumber': court_no,
        'MemberCode': member_code,
        'FullName': data.get('Name'),
        'District': data.get('District'),
        'Branch': data.get('Branch'),
        'Party': data.get('Party'),
        'Active': data.get('IsActive'),
        'LeadershipPosition': data.get('LeadershipPosition')
    }


async def main():
    async with aiohttp.ClientSession() as session:
        for branch in branches:
            for court_no in generalCourtNumber:

                #Get all members for the General Court
                url_members = f"https://malegislature.gov/api/GeneralCourts/{court_no}/LegislativeMembers"
                members_data, status = await fetch_json(session, url_members)

                # Handle non-success responses
                if status != 200 or members_data is None:
                    print(f"No member data found for General Court {court_no} for {branch}")
                    continue

                print(members_data)

                tasks = []

                #Combine both member and leadership info
                for m in members_data:
                    member_url = m.get('Details')
                    member_code = m.get('MemberCode')

                    tasks.append(
                        fetch_member_details(session, member_url, member_code, court_no)
                    )

                results = await asyncio.gather(*tasks)
                member_records.extend(results)

                print('Completed General Court ', court_no)

asyncio.run(main())


In [ ]:
member_records = pd.DataFrame(member_records)
member_records.to_csv('../../data/parsedcsv/MA_legislature_members_raw.csv', index=False)
print("Legislature members data saved to MA_legislature_members_raw.csv")
                      

In [ ]:
leg_mem_details= member_records

# Checking if a member has inconsistent details across general courts for name, district, branch or party
cols_to_check = ['FullName', 'District', 'Branch', 'Party']
inconsistent_members = (
    leg_mem_details
    .groupby('MemberCode')[cols_to_check]
    .nunique()  
    .reset_index()
)
# Filter members with more than one unique value in any of the specified columns
inconsistent_members = inconsistent_members[
    inconsistent_members[cols_to_check].gt(1).any(axis=1)
]
print(inconsistent_members)

In [ ]:
print(f"Total members extracted: {len(leg_mem_details)}")

## Bill-roll call mapping

In [ ]:
nest_asyncio.apply()

ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

# Function to fetch roll call data
async def fetch_rollcall(session, generalCourtNumber, billNumber, sem, retries=3):
    #Fetch roll call data for a given GeneralCourtNumber and BillNumber.
    url = f"https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/Documents/{billNumber}/RollCalls"

    # Semaphore to limit concurrent requests
    async with sem:
        for attempt in range(1, retries + 1):
            try:
                #  Make the GET request with timeout
                async with session.get(url, ssl=ssl_context, timeout=15) as resp:
                    if resp.status in (404, 400):
                        return []  # no data found for this bill
                    resp.raise_for_status()
                    data = await resp.json()

                    # Extract roll call numbers
                    rows = []
                    for item in data:
                        roll_call = item.get("RollCallNumber")
                        # Only add if roll call number exists
                        if roll_call is not None:
                            rows.append({
                                "GeneralCourtNumber": generalCourtNumber,
                                "BillNumber": billNumber,
                                "RollCallNumber": roll_call
                            })
                    return rows

            except Exception as e:
                print(f" Error {generalCourtNumber}-{billNumber} attempt {attempt}: {e}")
                await asyncio.sleep(2 * attempt)
        return []


async def get_bill_rollcall_map(df, max_concurrent=50):
    sem = asyncio.Semaphore(max_concurrent)
    async with aiohttp.ClientSession() as session:
        tasks = [
            fetch_rollcall(session, row.GeneralCourtNumber, row.BillNumber, sem)
            for _, row in df.iterrows()
        ]
        results = await tqdm_asyncio.gather(*tasks)
    
    return [i for sublist in results for i in sublist]


In [ ]:
# For the extracted bills of General Court, get the roll call mapping for each bill
bill_details = pd.read_csv("../../data/parsedcsv/MA_Bill_Details_raw.csv")
bill_details = bill_details.dropna(subset=["BillNumber"])

results_flat = await get_bill_rollcall_map(bill_details, max_concurrent=50)
rollcall_df = pd.DataFrame(results_flat)
rollcall_df.to_csv("../../data/parsedcsv/MA_Bill_rollcall_mapping.csv", index=False)

## Committee Data

In [ ]:
# This code block extracts committee details for each general court from 182 to 194
# 'https://malegislature.gov/api/GeneralCourts/{court_no}/Committees' API is used to extract committee links for each general court

committee=[]
for court_no in range(182, 195):
    # Extracting committee links for each general court
    committe_url = f'https://malegislature.gov/api/GeneralCourts/{court_no}/Committees'
    resp= requests.get(committe_url)
    resp.raise_for_status()
    items= resp.json()

    # Iterate through each committee link
    for i in items:
        url= i.get('Details')

        if not url:
            print(f'Skipping invalid URL for {court_no} committee')
            continue 

        # for each committee of general court
        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()  
            data = resp.json()

            docs = data.get('DocumentsBeforeCommittee', [])
            bills = []

            for d in docs: 
                bills.append(d.get('BillNumber'))
                
            # Append committee details to the list
            committee.append({
                'CommitteeCode': data.get('CommitteeCode'),
                'CommitteName': data.get('FullName'),
                'CommitteShortName': data.get('ShortName'),
                'GeneralCourtNumber': data.get('GeneralCourtNumber'),
                'Branch': data.get('Branch'),
                'SenateChairId': (data.get('SenateChairperson') or {}).get('MemberCode'),
                'HouseChairId': (data.get('HouseChairperson') or {}).get('MemberCode'),
                'Bills': bills
            })
            print(f'Successfully fetched committee data for Court {court_no} Committee {data.get("CommitteeCode")}')
        except Exception as e:
            print(f'Error while fetching {url}: {e}')




In [ ]:
committee_df = pd.DataFrame(committee)
committee_df.to_csv('../../data/parsedcsv/MA_committees.csv', index=False)
print("Committee data saved to MA_committees.csv")

In [ ]:
committee_df.head()